# `__slots__` и исключения в ООП – теория

## 1. `__slots__` – оптимизация памяти и ограничение атрибутов

По умолчанию каждый объект Python имеет словарь `__dict__`, в котором хранятся его атрибуты. Это удобно, но требует дополнительной памяти. `__slots__` позволяет:

- **Фиксировать набор атрибутов** – объекты с `__slots__` не имеют `__dict__`
- **Экономить память** – атрибуты хранятся не в словаре, а в дескрипторах (как в C-структуре)
- **Ускорить доступ** – доступ к атрибутам через `__slots__` быстрее

**Синтаксис:**

```python
class Point:
    __slots__ = ('x', 'y')
    def __init__(self, x, y):
        self.x = x
        self.y = y

**Важные ограничения:**

- Нельзя динамически добавлять новые атрибуты (если только `__slots__` не включает `'__dict__'`)
- Множественное наследование с `__slots__` требует осторожности (каждый класс определяет свои `__slots__`)
- Объекты с `__slots__` не поддерживают слабые ссылки, если не добавить `'__weakref__'` в `__slots__`
- При наследовании дочерний класс получает `__dict__`, если в его `__slots__` не перечислены атрибуты родителя

## 2. Исключения в ООП

Исключения – механизм для обработки ошибок и нештатных ситуаций.

### Иерархия

- `BaseException` – корень всех исключений
  - `SystemExit`, `KeyboardInterrupt` – системные
  - `Exception` – базовый для всех обычных ошибок
    - `ValueError`, `TypeError`, `IndexError`, `ArithmeticError` и т.д.

**Пользовательские исключения** создаются наследованием от `Exception` (или его подклассов):

In [ ]:
class ValidationError(Exception):
    pass

In [ ]:
try:
    # код, который может выбросить исключение
except ValueError as e:
    # обработка конкретного исключения
except (ZeroDivisionError, TypeError):
    # обработка нескольких типов
except:
    # плохая практика – перехватывает всё, включая SystemExit
else:
    # выполняется, если исключения не было
finally:
    # выполняется всегда (закрытие ресурсов)

## 🟢 Базовые задачи

### Задача 1. Класс с `__slots__`
Создайте класс `Student` с `__slots__ = ('name', 'age', 'grade')`. Реализуйте конструктор и метод `__str__`. Создайте несколько студентов. Попробуйте динамически добавить атрибут `extra`. Выведите размер объекта через `sys.getsizeof()` и сравните с классом без `__slots__`.

In [ ]:
# Шаблон для задачи 1
import sys

class Student:
    __slots__ = ...  # укажите атрибуты
    def __init__(self, name, age, grade):
        # инициализация
        pass

    def __str__(self):
        # вернуть строковое представление
        pass

# Создание объектов
s1 = Student("Alice", 20, 95)
# Попытка добавить атрибут
# s1.extra = "test"  # раскомментировать для проверки

# Вывод размера
print(sys.getsizeof(s1))

# Класс без __slots__ для сравнения
class RegularStudent:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self.grade = grade

r1 = RegularStudent("Bob", 21, 90)
print(sys.getsizeof(r1))  # плюс размер __dict__
print(sys.getsizeof(r1.__dict__))

### Задача 2. Пользовательское исключение
Создайте класс `BankAccount`. Реализуйте методы `deposit(amount)` и `withdraw(amount)`. Определите пользовательские исключения: `NegativeAmountError` (наследник `ValueError`) и `InsufficientFundsError` (наследник `ValueError`). При попытке внести отрицательную сумму или снять больше баланса выбрасывайте соответствующие исключения. В основном коде обработайте их.

In [ ]:
# Шаблон для задачи 2
class NegativeAmountError(ValueError):
    pass

class InsufficientFundsError(ValueError):
    pass

class BankAccount:
    def __init__(self, balance=0):
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount < 0:
            raise ...
        self._balance += amount

    def withdraw(self, amount):
        if amount < 0:
            raise ...
        if amount > self._balance:
            raise ...
        self._balance -= amount

# Проверка
acc = BankAccount(100)
try:
    acc.withdraw(200)
except InsufficientFundsError as e:
    print(e)

### Задача 3. Наследование исключений
Создайте иерархию исключений для библиотеки: `LibraryError` → `BookNotFoundError`, `BookAlreadyBorrowedError`. Класс `Library` имеет методы `add_book(title)`, `borrow_book(title)`, `return_book(title)`. При отсутствии книги – `BookNotFoundError`, при попытке взять уже выданную – `BookAlreadyBorrowedError`.

In [ ]:
# Шаблон для задачи 3
class LibraryError(Exception):
    pass

class BookNotFoundError(LibraryError):
    pass

class BookAlreadyBorrowedError(LibraryError):
    pass

class Library:
    def __init__(self):
        self._books = {}  # title -> borrowed (bool)

    def add_book(self, title):
        self._books[title] = False

    def borrow_book(self, title):
        if title not in self._books:
            raise ...
        if self._books[title]:
            raise ...
        self._books[title] = True

    def return_book(self, title):
        if title not in self._books:
            raise ...
        self._books[title] = False

# Демонстрация
lib = Library()
lib.add_book("1984")
try:
    lib.borrow_book("1984")
    lib.borrow_book("1984")
except BookAlreadyBorrowedError as e:
    print(e)

## 🔴 Задачи повышенной сложности

### Задача 4. `__slots__` и наследование
Создайте базовый класс `Shape` с `__slots__ = ('color',)`. Производный класс `Circle` наследует `Shape` и добавляет `__slots__ = ('radius',)`. Убедитесь, что экземпляры `Circle` не имеют `__dict__`. Проверьте, можно ли добавить атрибут `test`. Измерьте память для 1 млн кругов с помощью `tracemalloc` или `sys.getsizeof` списка. Сравните с классом без `__slots__`.

In [ ]:
# Шаблон для задачи 4
import sys
import tracemalloc

class Shape:
    __slots__ = ('color',)
    def __init__(self, color):
        self.color = color

class Circle(Shape):
    __slots__ = ('radius',)
    def __init__(self, color, radius):
        super().__init__(color)
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

# Создание 1 млн кругов
tracemalloc.start()
circles = [Circle("red", i) for i in range(1_000_000)]
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Память с __slots__: {current / 1024**2:.2f} MB")

# Для сравнения – класс Circle без __slots__
class RegularCircle(Shape):
    def __init__(self, color, radius):
        super().__init__(color)
        self.radius = radius
# Запустить аналогично и сравнить

### Задача 5. Исключения в магических методах
Создайте класс `LimitedList`, который при инициализации принимает `max_size`. Реализуйте `__setitem__(index, value)` – если индекс выходит за пределы (>= max_size), бросайте `ListFullError` (наследник `IndexError`). Если индекс больше текущей длины (но меньше max_size), бросайте обычный `IndexError`. Реализуйте `__getitem__` и `__len__`. В основной программе отловите оба типа ошибок по отдельности.

In [ ]:
# Шаблон для задачи 5
class ListFullError(IndexError):
    pass

class LimitedList:
    def __init__(self, max_size):
        self._max_size = max_size
        self._data = [None] * max_size
        self._length = 0

    def __setitem__(self, index, value):
        if index >= self._max_size:
            raise ...
        if index >= self._length and index != self._length:
            raise ...   # обычный IndexError
        self._data[index] = value
        if index == self._length:
            self._length += 1

    def __getitem__(self, index):
        if index < 0 or index >= self._length:
            raise ...
        return self._data[index]

    def __len__(self):
        return self._length

# Демонстрация
lst = LimitedList(3)
lst[0] = 10
lst[1] = 20
try:
    lst[3] = 30
except ListFullError as e:
    print(e)

## 🏠 Домашнее задание

### Задача 6. `__slots__` во вложенном классе
Создайте класс `Point3D` с `__slots__ = ('x', 'y', 'z')` и методом `distance_to(other)`. Создайте класс `PointCloud`, который хранит список точек. У `PointCloud` не должно быть `__slots__` (или может быть – решите сами). Реализуйте `add_point(x, y, z)`. Измерьте память для 100 000 точек (через `tracemalloc`). Напишите вторую версию `Point3D` без `__slots__` и сравните.

In [ ]:
# Шаблон для задачи 6
import tracemalloc

class Point3D:
    __slots__ = ('x', 'y', 'z')
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

    def distance_to(self, other):
        # вычислить евклидово расстояние
        pass

class PointCloud:
    def __init__(self):
        self.points = []

    def add_point(self, x, y, z):
        self.points.append(Point3D(x, y, z))

# Измерение памяти
tracemalloc.start()
cloud = PointCloud()
for i in range(100_000):
    cloud.add_point(i, i*2, i*3)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Память: {current / 1024**2:.2f} MB")

### Задача 7. Исключения и контекстный менеджер
Создайте контекстный менеджер `IgnoringErrors`, который принимает в конструкторе кортеж типов исключений. В `__exit__` подавляет эти исключения (возвращает `True`). Продемонстрируйте работу: внутри блока `with` несколько ошибок, но они игнорируются, программа продолжает выполнение.

In [ ]:
# Шаблон для задачи 7
class IgnoringErrors:
    def __init__(self, *exceptions):
        self.exceptions = exceptions

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exceptions):
            # подавить исключение
            return ...
        return False

# Демонстрация
with IgnoringErrors(ValueError, ZeroDivisionError):
    x = int("abc")
    y = 1 / 0
print("Программа продолжается")

### Задача 8. Комбинация `__slots__` и исключений
Создайте класс `ImmutablePoint` с `__slots__ = ('_x', '_y')`. Сделайте атрибуты `x` и `y` доступными только для чтения через `@property`. Переопределите `__setattr__` так, чтобы при попытке изменить `_x` или `_y` после инициализации выбрасывалось `AttributeError` с сообщением "Immutable attribute". Покажите работу.

In [ ]:
# Шаблон для задачи 8
class ImmutablePoint:
    __slots__ = ('_x', '_y')
    def __init__(self, x, y):
        self._x = x
        self._y = y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __setattr__(self, name, value):
        # Если атрибут уже существует и он в __slots__ – запретить изменение
        if name in self.__slots__ and hasattr(self, name):
            raise AttributeError(f"Immutable attribute: {name}")
        super().__setattr__(name, value)

# Проверка
p = ImmutablePoint(3, 4)
print(p.x, p.y)
# p.x = 5  # AttributeError